# NASA C-MAPSS: Individual vs Combined Model Comparison

**Goal:** Compare **Random Forest, Extra Trees, and XGBoost** trained (1) separately on FD001–FD004 and (2) once on all four datasets together. Choose the deployment approach from **validation MAE**, then perform a one-time final NASA test evaluation.

**Important:** FD001 numbers from the earlier notebook may change because this experiment uses a **consistent feature set** (all 21 sensors, 3 operating settings, cycles, and dataset indicators) across all four datasets. The earlier FD001 model removed six sensors. Don't compare those earlier results directly with this table.

**Evaluation:** Validation groups are split by engine, not row. NASA test scores use **only the last observed cycle per engine**. Select models and deployment approach **before** viewing test results.

## 1. Imports and paths
Run this notebook from your repository’s `notebooks/` folder, or from the repo root. Ensure the DVC-managed Parquet files are available locally.

In [ ]:
from pathlib import Path
import gc
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# Find the repository root, whether this notebook runs from notebooks/ or the root.
ROOT = Path("..").resolve() if Path("../data/curated").exists() else Path(".").resolve()
DATA_DIR = ROOT / "data" / "curated"
REPORT_DIR = ROOT / "reports" / "model_comparison"
MODEL_DIR = ROOT / "models" / "champion"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Keep every experiment reproducible.
SEED = 42
DATASETS = ["FD001", "FD002", "FD003", "FD004"]
ALGORITHMS = ["Random Forest", "Extra Trees", "XGBoost"]
print("Dataset folder:", DATA_DIR)
print("Data available:", DATA_DIR.exists())

## 2. Load all four datasets and use one consistent feature schema
Each row has a dataset indicator, so the combined model can distinguish FD001 from FD004. The four indicators are also included when fitting individual models (they are constant within each individual dataset).

In [ ]:
# Load curated training and held-out NASA test Parquet data.
train_data = {fd: pd.read_parquet(DATA_DIR / "train" / f"train_{fd}.parquet") for fd in DATASETS}
test_data = {fd: pd.read_parquet(DATA_DIR / "test" / f"test_{fd}.parquet") for fd in DATASETS}

# Use the same original feature columns for all models, including all 21 sensors.
base_features = [c for c in train_data["FD001"].columns if c not in ["unit_number", "RUL"]]
indicator_features = [f"dataset_{fd}" for fd in DATASETS]
feature_columns = base_features + indicator_features

# Check schemas before combining data; incorrect feature alignment is dangerous.
for fd in DATASETS:
    assert set(base_features).issubset(train_data[fd].columns), f"Missing train features: {fd}"
    assert set(base_features).issubset(test_data[fd].columns), f"Missing test features: {fd}"
    assert "RUL" in train_data[fd].columns and "RUL" in test_data[fd].columns, f"Missing RUL: {fd}"
    for name in DATASETS:
        train_data[fd][f"dataset_{name}"] = int(fd == name)
        test_data[fd][f"dataset_{name}"] = int(fd == name)
    assert not train_data[fd][feature_columns].isna().any().any(), f"Missing train values: {fd}"
    assert not test_data[fd][feature_columns].isna().any().any(), f"Missing test values: {fd}"
    print(fd, "train rows:", len(train_data[fd]), "test rows:", len(test_data[fd]), "train engines:", train_data[fd]["unit_number"].nunique())

print("Features per model:", len(feature_columns))

## 3. Split training engines once, and reuse exactly these splits
For example, FD001 contains 100 training engines: approximately 80 engines for fitting and 20 unseen engines for validation. **The combined model receives exactly the same training and validation rows** as the individual models.

In [ ]:
# Split each FD dataset by engine ID, not by cycle/row.
train_splits = {}
val_splits = {}
for fd in DATASETS:
    df = train_data[fd]
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(splitter.split(df, groups=df["unit_number"]))
    train_splits[fd] = df.iloc[train_idx].copy()
    val_splits[fd] = df.iloc[val_idx].copy()
    train_engines = set(train_splits[fd]["unit_number"])
    val_engines = set(val_splits[fd]["unit_number"])
    assert train_engines.isdisjoint(val_engines), f"Engine leakage: {fd}"
    print(fd, "training engines:", len(train_engines), "validation engines:", len(val_engines))

# Pool only the training portions for combined training; pool only the validation portions for validation.
combined_train = pd.concat([train_splits[fd] for fd in DATASETS], ignore_index=True)
combined_val = pd.concat([val_splits[fd] for fd in DATASETS], ignore_index=True)
print("Combined train rows:", len(combined_train), "Combined validation rows:", len(combined_val))

## 4. Model settings and error metric
All three algorithms use the same seed, splits, columns, and metric. **Lowest validation MAE wins**; RMSE is a secondary comparison metric. Each model is retrained once on its assigned training portion.

In [ ]:
# Create a new estimator each time so experiments never share trained state.
def make_model(name):
    if name == "Random Forest":
        return RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
    if name == "Extra Trees":
        return ExtraTreesRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
    if name == "XGBoost":
        return XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=SEED, n_jobs=-1, objective="reg:squarederror")
    raise ValueError(f"Unknown algorithm: {name}")

# Return RUL errors measured in cycles.
def evaluate(y_true, predictions):
    return {"mae": mean_absolute_error(y_true, predictions), "rmse": mean_squared_error(y_true, predictions) ** 0.5}

## 5. Train 12 individual models (3 algorithms × 4 datasets)
Keep the **best validation model** for each FD. Free the nonwinning estimators from memory as we go.

In [ ]:
individual_results = []
individual_winners = {}

# Repeat the same model comparison independently for each FD dataset.
for fd in DATASETS:
    train_df = train_splits[fd]
    val_df = val_splits[fd]
    best_mae = float("inf")
    for algorithm in ALGORITHMS:
        model = make_model(algorithm)
        model.fit(train_df[feature_columns], train_df["RUL"])
        pred = model.predict(val_df[feature_columns])
        scores = evaluate(val_df["RUL"], pred)
        individual_results.append({"dataset": fd, "algorithm": algorithm, "val_mae": scores["mae"], "val_rmse": scores["rmse"]})
        print(fd, algorithm, "MAE:", round(scores["mae"], 3), "RMSE:", round(scores["rmse"], 3))
        if scores["mae"] < best_mae:
            best_mae = scores["mae"]
            individual_winners[fd] = {"algorithm": algorithm, "model": model}
        else:
            del model
        gc.collect()

# Show the validation results for all 12 models.
individual_table = pd.DataFrame(individual_results).sort_values(["dataset", "val_mae"])
display(individual_table.round(3))

## 6. Train 3 combined models (all four datasets)
Each candidate is trained once on all available **training groups** and evaluated on all **validation groups**. The `dataset_FD...` feature columns identify the dataset of each row.

In [ ]:
combined_results = []
combined_winner = None
best_combined_mae = float("inf")

# Compare the same three algorithms on the pooled training data.
for algorithm in ALGORITHMS:
    model = make_model(algorithm)
    model.fit(combined_train[feature_columns], combined_train["RUL"])
    pred = model.predict(combined_val[feature_columns])
    scores = evaluate(combined_val["RUL"], pred)
    combined_results.append({"algorithm": algorithm, "val_mae": scores["mae"], "val_rmse": scores["rmse"]})
    print("Combined", algorithm, "MAE:", round(scores["mae"], 3), "RMSE:", round(scores["rmse"], 3))
    if scores["mae"] < best_combined_mae:
        best_combined_mae = scores["mae"]
        combined_winner = {"algorithm": algorithm, "model": model}
    else:
        del model
    gc.collect()

# Show all three combined-model results.
combined_table = pd.DataFrame(combined_results).sort_values("val_mae")
display(combined_table.round(3))

## 7. Fair validation comparison: 4 specialists vs 1 generalist
The four specialist models each predict **their own FD validation engines**. The combined model predicts those **same exact validation records**. Pool predictions before computing the overall MAE/RMSE; do **not** average four MAEs equally, because FD datasets have different numbers of rows.

In [ ]:
# Gather predictions for the exact same validation rows from both approaches.
individual_actual = []
individual_predictions = []
combined_predictions = []
by_dataset_validation = []
for fd in DATASETS:
    val_df = val_splits[fd]
    actual = val_df["RUL"].to_numpy()
    individual_pred = individual_winners[fd]["model"].predict(val_df[feature_columns])
    combined_pred = combined_winner["model"].predict(val_df[feature_columns])
    individual_actual.extend(actual)
    individual_predictions.extend(individual_pred)
    combined_predictions.extend(combined_pred)
    specialist_scores = evaluate(actual, individual_pred)
    generalist_scores = evaluate(actual, combined_pred)
    by_dataset_validation.append({"dataset": fd, "specialist": individual_winners[fd]["algorithm"], "specialist_mae": specialist_scores["mae"], "specialist_rmse": specialist_scores["rmse"], "combined_mae": generalist_scores["mae"], "combined_rmse": generalist_scores["rmse"]})

# Compute pooled validation errors across every held-out training-engine record.
individual_score = evaluate(individual_actual, individual_predictions)
combined_score = evaluate(individual_actual, combined_predictions)
approach_comparison = pd.DataFrame([{"approach": "Four individual models", "val_mae": individual_score["mae"], "val_rmse": individual_score["rmse"]}, {"approach": "One combined model", "val_mae": combined_score["mae"], "val_rmse": combined_score["rmse"]}]).sort_values("val_mae")
display(pd.DataFrame(by_dataset_validation).round(3))
display(approach_comparison.round(3))

# Lock the approach using validation MAE only: lower is better.
chosen_approach = "individual" if individual_score["mae"] <= combined_score["mae"] else "combined"
print("SELECTED DEPLOYMENT APPROACH:", chosen_approach)
print("Specialists:", {fd: individual_winners[fd]["algorithm"] for fd in DATASETS})
print("Combined candidate:", combined_winner["algorithm"])

# Save comparisons for the CV/README writeup.
individual_table.to_csv(REPORT_DIR / "all_individual_validation_results.csv", index=False)
combined_table.to_csv(REPORT_DIR / "all_combined_validation_results.csv", index=False)
pd.DataFrame(by_dataset_validation).to_csv(REPORT_DIR / "validation_by_dataset.csv", index=False)
approach_comparison.to_csv(REPORT_DIR / "validation_approach_comparison.csv", index=False)

## 8. Final held-out NASA test evaluation — run only after Section 7
**Do not change model settings based on these results.** NASA FD test labels correspond to the **last observed cycle per engine**, so this test computes one prediction per engine. For reporting transparency, we evaluate both already-selected approaches, but the deployment choice stays locked from validation.

In [ ]:
# Evaluate only the last observed record for every NASA test engine.
test_rows = {fd: test_data[fd].loc[test_data[fd].groupby("unit_number")["time_in_cycles"].idxmax()].copy() for fd in DATASETS}
test_results = []
all_test_actual = []
all_individual_test_predictions = []
all_combined_test_predictions = []

# Apply both previously chosen approaches to the same held-out engines.
for fd in DATASETS:
    df = test_rows[fd]
    actual = df["RUL"].to_numpy()
    individual_pred = individual_winners[fd]["model"].predict(df[feature_columns])
    combined_pred = combined_winner["model"].predict(df[feature_columns])
    scores_individual = evaluate(actual, individual_pred)
    scores_combined = evaluate(actual, combined_pred)
    test_results.append({"dataset": fd, "engines": len(df), "individual_mae": scores_individual["mae"], "individual_rmse": scores_individual["rmse"], "combined_mae": scores_combined["mae"], "combined_rmse": scores_combined["rmse"]})
    all_test_actual.extend(actual)
    all_individual_test_predictions.extend(individual_pred)
    all_combined_test_predictions.extend(combined_pred)

# Aggregate errors across all NASA test engines, not across average FD scores.
individual_test_score = evaluate(all_test_actual, all_individual_test_predictions)
combined_test_score = evaluate(all_test_actual, all_combined_test_predictions)
test_summary = pd.DataFrame([{"approach": "Four individual models", "test_mae": individual_test_score["mae"], "test_rmse": individual_test_score["rmse"]}, {"approach": "One combined model", "test_mae": combined_test_score["mae"], "test_rmse": combined_test_score["rmse"]}])
display(pd.DataFrame(test_results).round(3))
display(test_summary.round(3))
print("FINAL SELECTED APPROACH (chosen on validation only):", chosen_approach)

# Save the final, once-only held-out test comparison.
pd.DataFrame(test_results).to_csv(REPORT_DIR / "test_by_dataset.csv", index=False)
test_summary.to_csv(REPORT_DIR / "test_approach_comparison.csv", index=False)

## 9. Save the selected model(s) for the website
The future CSV-upload website will need 25 NASA input columns plus a dataset selection (FD001–FD004). If the specialists won, route each uploaded dataset to its corresponding model. If the generalist won, load the single combined model. The website must **not** request the `RUL` answer from users.

In [ ]:
# Export only the selected approach for a simple FastAPI deployment.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_files = {}
if chosen_approach == "combined":
    model_files["combined"] = "combined_model.joblib"
    joblib.dump(combined_winner["model"], MODEL_DIR / model_files["combined"])
else:
    for fd in DATASETS:
        model_files[fd] = f"{fd}_model.joblib"
        joblib.dump(individual_winners[fd]["model"], MODEL_DIR / model_files[fd])

# Save the exact required feature order and dataset indicators for API validation.
metadata = {"selected_approach": chosen_approach, "input_features": base_features, "model_features": feature_columns, "dataset_ids": DATASETS, "model_files": model_files, "selected_algorithms": {fd: individual_winners[fd]["algorithm"] for fd in DATASETS} if chosen_approach == "individual" else {"combined": combined_winner["algorithm"]}, "selection_metric": "pooled_validation_mae", "random_seed": SEED}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved selected model(s) and metadata to:", MODEL_DIR)
print("Selected approach:", chosen_approach)
print("Model files:", model_files)

### Notes for deployment and honest reporting

- **Validation comparison:** Full observed training trajectories for engine-held-out validation. **Test comparison:** Last observed test cycle per engine. These are different evaluation distributions, so explain them accurately.
- **No test tuning:** If the other approach has a lower test error, report it, but don't silently change the selected approach based on the test set.
- **Model registration:** Register the selected model(s) in MLflow **after reviewing the results**. This notebook first exports deployable `joblib` artifacts without requiring registry credentials.
- **CSV upload:** Require the dataset ID (FD001–FD004) on the website; the model cannot reliably infer the NASA dataset name from arbitrary user uploads. Ensure exactly the `input_features` listed in `metadata.json` are present. Add the dataset-indicator columns server-side.
- **Environment:** Add `xgboost` to `requirements.txt`, and keep `scikit-learn`, `pandas`, `pyarrow`, `joblib`, and their versions consistent between training and deployment.

## 10. Log completed experiments to DagsHub MLflow — **no retraining**

**You have already executed Sections 1–9.** Run **only the new cells below in your still-running notebook kernel**. They use the existing `individual_results`, `combined_results`, `individual_winners`, `combined_winner`, `individual_score`, `combined_score`, and test scores. They do **not** call `.fit()`.

- MLflow records **all 15 candidate runs** (12 specialists + 3 combined) with validation metrics and parameters.
- It logs the **five retained model artifacts** (the winning model for each FD and the winning combined model). The other 10 candidates were discarded by the original training loops, so their **metrics** can be logged but their **model binaries** are no longer available.
- A separate comparison run records pooled validation and test scores, plus report CSV artifacts.
- **Nothing is registered** in the Model Registry yet. Registration can happen after you confirm the deployment selection.

**Important:** Downloading and opening this notebook as a new file might start a fresh kernel. If so, **copy Sections 10–12 into the notebook where you have already run all the training cells**. This preserves the trained models and avoids rerunning training.

In [ ]:
import mlflow
import mlflow.sklearn
import dagshub

# Verify the existing notebook variables are still available; never retrain them here.
required = ["individual_results", "combined_results", "individual_winners", "combined_winner", "individual_score", "combined_score", "individual_test_score", "combined_test_score", "test_results", "chosen_approach", "feature_columns", "REPORT_DIR", "SEED", "DATASETS", "ALGORITHMS"]
missing = [name for name in required if name not in globals()]
assert not missing, f"Notebook variables missing: {missing}. Run these cells in your original active notebook kernel."
assert combined_winner is not None, "The winning combined model is missing from memory."
assert len(individual_results) == 12 and len(combined_results) == 3, "Expected results for 12 individual and 3 combined experiments."
assert all(fd in individual_winners for fd in DATASETS), "A winning individual model is missing."

# Connect experiment tracking to the existing DagsHub repository.
dagshub.init(repo_owner="Omesham", repo_name="predictive-maintenance-mlops", mlflow=True)
assert "dagshub.com" in mlflow.get_tracking_uri(), "MLflow is not pointing to DagsHub; check authentication before logging."
mlflow.set_experiment("CMAPSS_FD001_FD004_Individual_vs_Combined")
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Ready to log 15 experiment runs without retraining.")

## 11. Log the 15 model comparisons and available models

Run this cell **once**. It sends metrics for every candidate to MLflow and uploads the five existing fitted winner objects. This might take a little time because the retained tree ensembles need to be uploaded, but **no model is fitted again**. The logged model format uses `cloudpickle`; load model artifacts only from trusted sources.

In [ ]:
# Preserve the training settings used in Section 4 of this notebook.
def tracking_params(algorithm):
    params = {"random_state": SEED, "n_jobs": -1, "n_estimators": 300 if algorithm == "XGBoost" else 100}
    if algorithm == "XGBoost":
        params.update({"max_depth": 5, "learning_rate": 0.05, "objective": "reg:squarederror"})
    return params

# Look up previously computed held-out test scores; do not make new test predictions.
test_by_fd = {row["dataset"]: row for row in test_results}
logged_runs = []

# Log 12 individual candidates. Retained winners also get their fitted model artifacts.
for record in individual_results:
    fd = record["dataset"]
    algorithm = record["algorithm"]
    winner = individual_winners[fd]
    retained = winner["algorithm"] == algorithm
    with mlflow.start_run(run_name=f"{fd}_{algorithm.replace(' ', '_')}") as run:
        mlflow.set_tags({"approach": "individual", "dataset": fd, "algorithm": algorithm, "retained_model": str(retained).lower(), "selected_for_deployment": str(retained and chosen_approach == "individual").lower()})
        mlflow.log_params(tracking_params(algorithm))
        mlflow.log_metric("val_mae", float(record["val_mae"]))
        mlflow.log_metric("val_rmse", float(record["val_rmse"]))
        if retained:
            mlflow.log_metric("test_mae", float(test_by_fd[fd]["individual_mae"]))
            mlflow.log_metric("test_rmse", float(test_by_fd[fd]["individual_rmse"]))
            mlflow.sklearn.log_model(winner["model"], name="model", serialization_format="cloudpickle")
        logged_runs.append({"approach": "individual", "dataset": fd, "algorithm": algorithm, "run_id": run.info.run_id, "model_logged": retained})
    print("Logged:", fd, algorithm, "model artifact:", retained)

# Log 3 combined candidates. Only the combined winner's fitted model remains in memory.
for record in combined_results:
    algorithm = record["algorithm"]
    retained = combined_winner["algorithm"] == algorithm
    with mlflow.start_run(run_name=f"Combined_{algorithm.replace(' ', '_')}") as run:
        mlflow.set_tags({"approach": "combined", "dataset": "FD001+FD002+FD003+FD004", "algorithm": algorithm, "retained_model": str(retained).lower(), "selected_for_deployment": str(retained and chosen_approach == "combined").lower()})
        mlflow.log_params(tracking_params(algorithm))
        mlflow.log_metric("val_mae", float(record["val_mae"]))
        mlflow.log_metric("val_rmse", float(record["val_rmse"]))
        if retained:
            mlflow.log_metric("test_mae", float(combined_test_score["mae"]))
            mlflow.log_metric("test_rmse", float(combined_test_score["rmse"]))
            mlflow.sklearn.log_model(combined_winner["model"], name="model", serialization_format="cloudpickle")
        logged_runs.append({"approach": "combined", "dataset": "all_four", "algorithm": algorithm, "run_id": run.info.run_id, "model_logged": retained})
    print("Logged: Combined", algorithm, "model artifact:", retained)

# Keep the new MLflow run IDs in a local comparison report for traceability.
logged_runs_df = pd.DataFrame(logged_runs)
logged_runs_df.to_csv(REPORT_DIR / "mlflow_run_ids.csv", index=False)
print("Experiment runs logged:", len(logged_runs_df))
print("Fitted models uploaded:", int(logged_runs_df["model_logged"].sum()))

## 12. Log overall approach comparison and exported reports

This produces a **16th run** named `Individual_vs_Combined_Summary` that makes the deployment decision and final scores easy to inspect. It uploads the comparison CSVs that the original notebook already saved. It **does not register** a model.

In [ ]:
# Track the final approach comparison, using metrics computed in the earlier notebook cells.
with mlflow.start_run(run_name="Individual_vs_Combined_Summary") as summary_run:
    mlflow.set_tags({"run_type": "approach_comparison", "selected_approach": chosen_approach, "selection_basis": "pooled_validation_mae"})
    mlflow.log_param("datasets", ",".join(DATASETS))
    mlflow.log_param("algorithms", ",".join(ALGORITHMS))
    mlflow.log_param("random_seed", SEED)
    mlflow.log_metric("individual_val_mae", float(individual_score["mae"]))
    mlflow.log_metric("individual_val_rmse", float(individual_score["rmse"]))
    mlflow.log_metric("combined_val_mae", float(combined_score["mae"]))
    mlflow.log_metric("combined_val_rmse", float(combined_score["rmse"]))
    mlflow.log_metric("individual_test_mae", float(individual_test_score["mae"]))
    mlflow.log_metric("individual_test_rmse", float(individual_test_score["rmse"]))
    mlflow.log_metric("combined_test_mae", float(combined_test_score["mae"]))
    mlflow.log_metric("combined_test_rmse", float(combined_test_score["rmse"]))
    for filename in ["all_individual_validation_results.csv", "all_combined_validation_results.csv", "validation_by_dataset.csv", "validation_approach_comparison.csv", "test_by_dataset.csv", "test_approach_comparison.csv", "mlflow_run_ids.csv"]:
        file_path = REPORT_DIR / filename
        if file_path.is_file():
            mlflow.log_artifact(str(file_path), artifact_path="comparison_reports")

print("Summary run ID:", summary_run.info.run_id)
print("Selected using validation results:", chosen_approach)
print("Finished: 15 candidate runs + 1 summary run, without retraining.")